In [1]:
import pandas as pd
from pathlib import Path

# Point to the frozen canonical data from Phase A.
project_root = Path.home() / "Desktop" / "nba_economics"
canonical_dir = project_root / "data" / "canonical"

# Load only the two tables relevant to F1.
seasons = pd.read_csv(canonical_dir / "seasons.csv")
team_seasons = pd.read_csv(canonical_dir / "team_seasons.csv")

# Display row counts and column names.
for name, df in {
    "C01 — seasons": seasons,
    "C04 — team_seasons": team_seasons,
}.items():
    print(f"\n{name}: {df.shape[0]} rows")
    print(df.columns.tolist())


C01 — seasons: 26 rows
['season', 'Year', 'Salary Cap', 'Salary Cap Floor Pct', 'Salary Cap Floor', 'Luxury Tax', '1st Apron', '2nd Apron', 'BAE', 'Non-Taxpayer MLE', 'Taxpayer MLE', 'Team Room MLE', 'cba_id']

C04 — team_seasons: 776 rows
['team_name', 'wins', 'losses', 'win_pct', 'games_behind', 'points_per_game', 'opponent_points_per_game', 'simple_rating_system', 'conference', 'season', 'franchise_id', 'league_rank', 'average_age', 'pythagorean_wins', 'pythagorean_losses', 'margin_of_victory', 'strength_of_schedule', 'offensive_rating', 'defensive_rating', 'net_rating', 'pace', 'free_throw_rate', 'three_point_attempt_rate', 'true_shooting_pct', 'off_eFG%', 'off_TOV%', 'off_ORB%', 'off_FT/FGA', 'def_eFG%', 'def_TOV%', 'def_DRB%', 'def_FT/FGA', 'arena', 'attendance', 'attendance_per_game', 'seed', 'playoff_series_won', 'round_score', 'furthest_round', 'team_payroll', 'salary_cap_hit', 'salary_cap_space', 'made_playoffs', 's18_tax_threshold', 'taxable_payroll', 'tax_space', 'tax_paid

In [2]:
# Columns relevant to F1 in each canonical table.
c01_cols = [
    "season",
    "Salary Cap",
    "Salary Cap Floor",
    "Luxury Tax",
    "1st Apron",
    "2nd Apron",
    "cba_id",
]

c04_cols = [
    "season",
    "franchise_id",
    "team_payroll",
    "salary_cap_hit",
    "salary_cap_space",
    "taxable_payroll",
    "s18_tax_threshold",
    "tax_paid",
    "has_luxury_tax_source",
]

# Loop through the two tables and their relevant columns.
for name, df, columns in [
    ("C01 — seasons", seasons, c01_cols),
    ("C04 — team_seasons", team_seasons, c04_cols),
]:
    print(f"\n{'=' * 60}")
    print(name)

    # Calculate how much of each column is populated.
    coverage = (
        df[columns]
        .notna()
        .mean()
        .mul(100)
        .round(1)
    )

    print("\nCoverage (%):")
    print(coverage.to_string())

    # Display a few records so we can inspect actual values.
    print("\nSample rows:")
    display(df[columns].head(5))


C01 — seasons

Coverage (%):
season              100.0
Salary Cap          100.0
Salary Cap Floor    100.0
Luxury Tax           88.5
1st Apron            57.7
2nd Apron            11.5
cba_id              100.0

Sample rows:


,season,Salary Cap,Salary Cap Floor,Luxury Tax,1st Apron,2nd Apron,cba_id
0,2000-2001,35500000,26625000,NaN,NaN,NaN,cba_1999
1,2001-2002,42500000,31875000,NaN,NaN,NaN,cba_1999
2,2002-2003,40271000,30203250,52880000.0,NaN,NaN,cba_1999
3,2003-2004,43840000,32880000,54556722.0,NaN,NaN,cba_1999
4,2004-2005,43870000,32902500,NaN,NaN,NaN,cba_1999



C04 — team_seasons

Coverage (%):
season                   100.0
franchise_id             100.0
team_payroll             100.0
salary_cap_hit           100.0
salary_cap_space          11.6
taxable_payroll           58.0
s18_tax_threshold         88.7
tax_paid                  34.5
has_luxury_tax_source    100.0

Sample rows:


,season,franchise_id,team_payroll,salary_cap_hit,salary_cap_space,taxable_payroll,s18_tax_threshold,tax_paid,has_luxury_tax_source
0,2000-2001,1610612755,51085458,51085458,NaN,NaN,NaN,NaN,False
1,2000-2001,1610612748,73472329,73472329,NaN,NaN,NaN,NaN,False
2,2000-2001,1610612752,74007738,74007738,NaN,NaN,NaN,NaN,False
3,2000-2001,1610612753,37459860,37459860,NaN,NaN,NaN,NaN,False
4,2000-2001,1610612738,51478707,51478707,NaN,NaN,NaN,NaN,False


In [3]:
# Compare the different payroll measures across all franchise-seasons.

# Count rows where the two fully populated measures differ.
cap_hit_differs = (
    team_seasons["team_payroll"]
    != team_seasons["salary_cap_hit"]
)

print(
    "Team payroll vs salary cap hit — differing rows:",
    cap_hit_differs.sum()
)

# Restrict comparisons to rows with an observed taxable payroll.
tax_comparison = team_seasons.dropna(
    subset=["taxable_payroll"]
).copy()

# Calculate the difference in dollars.
tax_comparison["payroll_difference"] = (
    tax_comparison["taxable_payroll"]
    - tax_comparison["team_payroll"]
)

print("\nTaxable payroll vs team payroll:")
print(tax_comparison["payroll_difference"].describe())

print("\nExamples of largest differences:")
display(
    tax_comparison[
        [
            "season",
            "franchise_id",
            "team_payroll",
            "taxable_payroll",
            "payroll_difference",
        ]
    ]
    .assign(
        absolute_difference=lambda df:
        df["payroll_difference"].abs()
    )
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    .head(10)
)

Team payroll vs salary cap hit — differing rows: 0

Taxable payroll vs team payroll:
count    4.500000e+02
mean    -1.612971e+06
std      5.873964e+06
min     -2.999286e+07
25%     -3.000656e+06
50%     -1.294490e+06
75%      2.464650e+04
max      3.130280e+07
Name: payroll_difference, dtype: float64

Examples of largest differences:


,season,franchise_id,team_payroll,taxable_payroll,payroll_difference,absolute_difference
589,2019-2020,1610612763,98495848,129798650.0,31302802.0,31302802.0
545,2018-2019,1610612748,153171497,123178641.0,-29992856.0,29992856.0
755,2025-2026,1610612748,157325624,187027668.0,29702044.0,29702044.0
511,2017-2018,1610612748,133624374,105065917.0,-28558457.0,28558457.0
574,2019-2020,1610612766,96552033,124096301.0,27544268.0,27544268.0
578,2019-2020,1610612765,104527576,131251409.0,26723833.0,26723833.0
400,2013-2014,1610612753,79294567,53516731.0,-25777836.0,25777836.0
547,2018-2019,1610612737,79180081,103686836.0,24506755.0,24506755.0
575,2019-2020,1610612764,121296256,97584993.0,-23711263.0,23711263.0
590,2019-2020,1610612756,98539675,120782748.0,22243073.0,22243073.0


### F1 Audit Finding 01 — Payroll Measure Selection

**Observed results**
- `team_payroll` and `salary_cap_hit` have identical values across all 776 franchise-season observations.
- `taxable_payroll` is available for 450 observations and differs materially from `team_payroll` in some cases.
- The median difference (`taxable_payroll - team_payroll`) is approximately −$1.29 million.

**Feature-design decisions**
1. Use `team_payroll` as the default numerator for descriptive financial features, including payroll ratios, spending rankings, and year-over-year payroll changes.
2. Retain `taxable_payroll` separately for tax-specific financial analysis.
3. Do not assume that `team_payroll` accurately determines official luxury-tax or apron status.
4. Do not substitute `team_payroll` for missing `taxable_payroll` without explicitly labeling the resulting measure as a proxy.

**Status:** Default payroll numerator selected. Tax and apron compliance features remain conditional pending further validation.

In [4]:
# Attach the league-wide luxury-tax threshold to each franchise-season.
tax_audit = team_seasons.merge(
    seasons[["season", "Luxury Tax"]],
    on="season",
    how="left",
    validate="many_to_one"  # Many teams belong to one league season.
)

# Only compare rows where both threshold values exist.
both_present = tax_audit[
    tax_audit["s18_tax_threshold"].notna()
    & tax_audit["Luxury Tax"].notna()
].copy()

# Calculate differences between the two tax-threshold sources.
both_present["threshold_difference"] = (
    both_present["s18_tax_threshold"]
    - both_present["Luxury Tax"]
)

print("Rows with both tax thresholds:", len(both_present))
print(
    "Rows with differing thresholds:",
    (both_present["threshold_difference"].abs() > 1).sum()
)

# Show the actual seasons and differences if inconsistencies exist.
display(
    both_present.loc[
        both_present["threshold_difference"].abs() > 1,
        [
            "season",
            "franchise_id",
            "s18_tax_threshold",
            "Luxury Tax",
            "threshold_difference",
        ],
    ].head(20)
)

Rows with both tax thresholds: 688
Rows with differing thresholds: 208


,season,franchise_id,s18_tax_threshold,Luxury Tax,threshold_difference
58,2002-2003,1610612751,52900000.0,52880000.0,20000.0
59,2002-2003,1610612755,52900000.0,52880000.0,20000.0
60,2002-2003,1610612738,52900000.0,52880000.0,20000.0
61,2002-2003,1610612753,52900000.0,52880000.0,20000.0
62,2002-2003,1610612764,52900000.0,52880000.0,20000.0
63,2002-2003,1610612752,52900000.0,52880000.0,20000.0
64,2002-2003,1610612748,52900000.0,52880000.0,20000.0
65,2002-2003,1610612765,52900000.0,52880000.0,20000.0
66,2002-2003,1610612754,52900000.0,52880000.0,20000.0
67,2002-2003,1610612740,52900000.0,52880000.0,20000.0


In [5]:
# Focus only on observations where the two sources disagree.
mismatches = both_present.loc[
    both_present["threshold_difference"].abs() > 1
].copy()

# Summarize differences at the league-season level.
season_discrepancies = (
    mismatches.groupby("season")
    .agg(
        affected_teams=("franchise_id", "nunique"),
        c04_threshold=("s18_tax_threshold", "first"),
        c01_threshold=("Luxury Tax", "first"),
        difference=("threshold_difference", "first"),
        unique_c04_values=("s18_tax_threshold", "nunique"),
    )
    .reset_index()
)

# Calculate relative differences to distinguish rounding
# from potentially material inconsistencies.
season_discrepancies["difference_pct"] = (
    season_discrepancies["difference"].abs()
    / season_discrepancies["c01_threshold"]
    * 100
)

display(season_discrepancies)

,season,affected_teams,c04_threshold,c01_threshold,difference,unique_c04_values,difference_pct
0,2002-2003,29,52900000.0,52880000.0,20000.0,1,0.037821
1,2003-2004,29,54600000.0,54556722.0,43278.0,1,0.079327
2,2006-2007,30,65400000.0,65420000.0,-20000.0,1,0.030572
3,2007-2008,30,67900000.0,67865000.0,35000.0,1,0.051573
4,2008-2009,30,71200000.0,71150000.0,50000.0,1,0.070274
5,2009-2010,30,69900000.0,69920000.0,-20000.0,1,0.028604
6,2010-2011,30,70300000.0,70307000.0,-7000.0,1,0.009956


### F1 Audit Finding 02 — Luxury-Tax Threshold Consistency

**Observed results**
- Compared 688 franchise-season records with both C01 and C04 tax thresholds populated.
- Found 208 differing records, concentrated in seven seasons.
- Within each affected season, the C04 threshold was consistent across affected franchises.
- Absolute discrepancies ranged from $7,000 to $50,000, with a maximum relative difference of approximately 0.079%.

**Interpretation**

The differences are consistent with source rounding or precision differences. There is no evidence in this comparison of large discrepancies between the two sources.

**Feature-design decision**

Use C01 `Luxury Tax` as the default league-wide luxury-tax threshold for F1 financial features. Retain C04 `s18_tax_threshold` for reference and source-specific validation.

**Remaining caveat**

The exact historical source definitions and precision conventions have not been independently verified. Teams extremely close to the threshold may require additional checking.

**Status:** Provisionally ready for descriptive tax-threshold features; official tax-liability classifications remain conditional.

In [6]:
# Show the availability of financial thresholds across all seasons.
threshold_cols = [
    "Salary Cap Floor",
    "Salary Cap",
    "Luxury Tax",
    "1st Apron",
    "2nd Apron",
]

threshold_availability = seasons[
    ["season", "cba_id"] + threshold_cols
].copy()

# Convert each threshold into a Boolean indicating whether
# a value exists for that season.
threshold_availability[threshold_cols] = (
    threshold_availability[threshold_cols].notna()
)

display(threshold_availability)

,season,cba_id,Salary Cap Floor,Salary Cap,Luxury Tax,1st Apron,2nd Apron
0,2000-2001,cba_1999,True,True,False,False,False
1,2001-2002,cba_1999,True,True,False,False,False
2,2002-2003,cba_1999,True,True,True,False,False
3,2003-2004,cba_1999,True,True,True,False,False
4,2004-2005,cba_1999,True,True,False,False,False
5,2005-2006,cba_2005,True,True,True,False,False
6,2006-2007,cba_2005,True,True,True,False,False
7,2007-2008,cba_2005,True,True,True,False,False
8,2008-2009,cba_2005,True,True,True,False,False
9,2009-2010,cba_2005,True,True,True,False,False


### F1 Audit Finding 03 — Historical Threshold Applicability

**Observed results**
- Salary cap and salary floor: populated for all 26 seasons.
- Luxury-tax threshold: populated for 23 seasons, excluding 2000–01, 2001–02 and 2004–05.
- First apron: populated from 2011–12 onward.
- Second apron: populated from 2023–24 onward.
- Threshold availability aligns broadly with the historical development of NBA collective bargaining rules.

**Feature-design decisions**
1. Calculate threshold ratios and distances only where the corresponding threshold is available.
2. Preserve missing values for inapplicable or unavailable thresholds.
3. Distinguish historical CBA eras when analyzing apron-related behavior.
4. Treat payroll-based threshold crossing as descriptive rather than definitive evidence of official CBA status.

**Status:** F1 foundational financial features are feasible. Tax-specific and apron-related interpretations remain conditional on source definitions and CBA rules.

## F2 — Player Compensation and Value

In [7]:

# F2 | Step 1A — Inspect the salary and player-team-season tables.
# We are checking their structure, not building features yet.

player_salaries = pd.read_csv(canonical_dir / "player_salaries.csv")
player_team_seasons = pd.read_csv(
    canonical_dir / "player_team_seasons.csv"
)

# Store the two DataFrames in a dictionary so we can
# inspect them using one loop instead of repeating code.
f2_sources = {
    "C07 — Player Salaries": player_salaries,
    "C05 — Player-Team Seasons": player_team_seasons,
}

for name, df in f2_sources.items():

    print(f"\n{'=' * 60}")
    print(name)
    print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

    # Print the actual column names before choosing join keys.
    print("\nColumns:")
    print(df.columns.tolist())

    # Show the first three records to understand the data layout.
    print("\nSample records:")
    display(df.head(3))



C07 — Player Salaries
Rows: 11,826 | Columns: 6

Columns:
['player_id', 'franchise_id', 'season', 'salary', 'source_player_name', 'source_team_name']

Sample records:


,player_id,franchise_id,season,salary,source_player_name,source_team_name
0,389,1610612737,2000-2001,7167000,Toni Kukoc,Atlanta Hawks
1,673,1610612737,2000-2001,5909000,Alan Henderson,Atlanta Hawks
2,953,1610612737,2000-2001,4950000,Lorenzen Wright,Atlanta Hawks



C05 — Player-Team Seasons
Rows: 12,810 | Columns: 49

Columns:
['player_id', 'franchise_id', 'season', 'age', 'height_inches', 'weight_lbs', 'games_played', 'points_total', 'rebounds_total', 'assists_total', 'wins', 'losses', 'win_pct', 'minutes_per_game', 'field_goals_made_per_game', 'field_goals_attempted_per_game', 'field_goal_pct', 'three_pointers_made_per_game', 'three_pointers_attempted_per_game', 'three_point_pct', 'free_throws_made_per_game', 'free_throws_attempted_per_game', 'free_throw_pct', 'offensive_rebounds_per_game', 'defensive_rebounds_per_game', 'rebounds_per_game', 'assists_per_game', 'turnovers_per_game', 'steals_per_game', 'blocks_per_game', 'personal_fouls_per_game', 'points_per_game', 'plus_minus_per_game', 'offensive_rating', 'defensive_rating', 'net_rating', 'assist_pct', 'assist_to_turnover_ratio', 'assist_ratio', 'offensive_rebound_pct', 'defensive_rebound_pct', 'rebound_pct', 'team_turnover_pct', 'effective_field_goal_pct', 'true_shooting_pct', 'usage_pct', 

,player_id,franchise_id,season,age,height_inches,weight_lbs,games_played,points_total,rebounds_total,assists_total,...,offensive_rebound_pct,defensive_rebound_pct,rebound_pct,team_turnover_pct,effective_field_goal_pct,true_shooting_pct,usage_pct,pace,player_impact_estimate,possessions
0,920,1610612748,2000-2001,37.0,81.0,225,82,367,313,39,...,0.076,0.151,0.113,9.9,0.444,0.492,0.137,90.37,0.093,2660
1,2062,1610612741,2000-2001,23.0,73.0,180,33,198,36,64,...,0.016,0.046,0.030,8.3,0.477,0.495,0.167,90.19,0.071,1173
2,243,1610612755,2000-2001,28.0,77.0,209,76,878,311,377,...,0.015,0.115,0.066,14.7,0.511,0.549,0.186,93.17,0.119,4639


In [8]:

# F2 | Step 1B — Measure completeness of each source column.
# A missing salary or identifier can affect downstream feature reliability.

for name, df in f2_sources.items():

    print(f"\n{name} — Column coverage")

    coverage = (
        df.notna()
        .mean()            # Fraction of rows with non-null values
        .mul(100)          # Convert fractions into percentages
        .round(1)         # Round to one decimal place
        .sort_values()     # Least-complete columns appear first
    )

    display(coverage.to_frame(name="coverage_pct"))



C07 — Player Salaries — Column coverage


,coverage_pct
player_id,100.0
franchise_id,100.0
season,100.0
salary,100.0
source_player_name,100.0
source_team_name,100.0



C05 — Player-Team Seasons — Column coverage


,coverage_pct
height_inches,99.5
weight_lbs,99.9
player_id,100.0
turnovers_per_game,100.0
steals_per_game,100.0
blocks_per_game,100.0
personal_fouls_per_game,100.0
points_per_game,100.0
plus_minus_per_game,100.0
offensive_rating,100.0


In [9]:

# Demonstrate the same coverage calculation using one column.

column = player_team_seasons["height_inches"]

total_rows = len(column)
nonmissing_rows = column.notna().sum()
missing_rows = column.isna().sum()

# Calculate coverage using explicit numerator / denominator.
coverage_pct = (nonmissing_rows / total_rows) * 100

print(f"Total rows: {total_rows:,}")
print(f"Nonmissing height values: {nonmissing_rows:,}")
print(f"Missing height values: {missing_rows:,}")
print(f"Coverage: {coverage_pct:.1f}%")


Total rows: 12,810
Nonmissing height values: 12,741
Missing height values: 69
Coverage: 99.5%


In [10]:

# F2 | Step 2A — Test the proposed player-franchise-season grain.
# A composite key combines multiple columns to identify one observation.

player_team_key = ["player_id", "franchise_id", "season"]

for name, df in f2_sources.items():

    # duplicated() returns True for repeated combinations of these keys.
    # keep=False marks every row belonging to a duplicate group.
    duplicate_rows = df.duplicated(
        subset=player_team_key,
        keep=False
    )

    print(f"\n{name}")
    print(f"Total rows: {len(df):,}")
    print(f"Rows involved in duplicate keys: {duplicate_rows.sum():,}")

    # Display examples only if duplicate combinations exist.
    if duplicate_rows.any():
        display(
            df.loc[duplicate_rows]
            .sort_values(player_team_key)
            .head(10)
        )



C07 — Player Salaries
Total rows: 11,826
Rows involved in duplicate keys: 0

C05 — Player-Team Seasons
Total rows: 12,810
Rows involved in duplicate keys: 0


In [11]:

# F2 | Step 2B — Compare player-franchise-season keys across C05 and C07.

player_team_key = ["player_id", "franchise_id", "season"]

salary_key_audit = player_team_seasons[player_team_key].merge(
    player_salaries[player_team_key],

    # Match on all three fields, not just player_id.
    on=player_team_key,

    # Keep all keys from both tables, including unmatched records.
    how="outer",

    # Add a column showing where each merged row originated.
    indicator=True,

    # Both tables passed the uniqueness check, so this should be 1:1.
    validate="one_to_one"
)

# Count matched and unmatched records.
match_counts = salary_key_audit["_merge"].value_counts()

display(match_counts)


_merge
both          10601
left_only      2209
right_only     1225
Name: count, dtype: int64

In [12]:

# F2 | Step 2C — Examine salary matching by season.
# This helps identify whether missing matches cluster in certain years.

season_match_counts = (
    salary_key_audit
    .groupby(["season", "_merge"], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Calculate the share of C05 records that have matching salary records.
season_match_counts["c05_match_pct"] = (
    season_match_counts["both"]
    / (
        season_match_counts["both"]
        + season_match_counts["left_only"]
    )
    * 100
).round(1)

display(season_match_counts)


_merge,left_only,right_only,both,c05_match_pct
season,,,,
2000-2001,72,1,369,83.7
2001-2002,29,1,411,93.4
2002-2003,150,0,278,65.0
2003-2004,265,2,177,40.0
2004-2005,346,1,118,25.4
2005-2006,38,9,420,91.7
2006-2007,26,26,432,94.3
2007-2008,22,41,429,95.1
2008-2009,18,34,427,96.0


In [13]:

# F2 | Step 2D — Diagnose unmatched records at the player-season level.
# A player may appear in both datasets but under different franchises.

# Select keys that failed to match across C05 and C07.
unmatched = salary_key_audit.loc[
    salary_key_audit["_merge"] != "both"
].copy()

# Create sets of player-season combinations from each source.
# Sets allow fast membership checks without constructing another merge.
c05_player_seasons = set(
    zip(player_team_seasons["player_id"], player_team_seasons["season"])
)

c07_player_seasons = set(
    zip(player_salaries["player_id"], player_salaries["season"])
)

# For unmatched C05 records, check whether the player-season
# exists in C07 under some other franchise.
c05_unmatched = unmatched.loc[
    unmatched["_merge"] == "left_only"
].copy()

c05_unmatched["exists_in_c07_other_team"] = [
    (player_id, season) in c07_player_seasons
    for player_id, season in zip(
        c05_unmatched["player_id"],
        c05_unmatched["season"]
    )
]

# For unmatched C07 records, perform the reverse check.
c07_unmatched = unmatched.loc[
    unmatched["_merge"] == "right_only"
].copy()

c07_unmatched["exists_in_c05_other_team"] = [
    (player_id, season) in c05_player_seasons
    for player_id, season in zip(
        c07_unmatched["player_id"],
        c07_unmatched["season"]
    )
]

print("C05-only records:")
print(c05_unmatched["exists_in_c07_other_team"].value_counts())

print("\nC07-only records:")
print(c07_unmatched["exists_in_c05_other_team"].value_counts())


C05-only records:
exists_in_c07_other_team
False    2113
True       96
Name: count, dtype: int64

C07-only records:
exists_in_c05_other_team
False    1128
True       97
Name: count, dtype: int64


In [14]:

# F2 | Step 3A — Reconcile observed player salaries with team payroll.
# This is an audit calculation, not a saved analytical feature.

# Sum all observed C07 salaries within each franchise-season.
salary_totals = (
    player_salaries
    .groupby(["franchise_id", "season"], as_index=False)
    .agg(
        observed_salary_total=("salary", "sum"),
        salary_record_count=("salary", "size")
    )
)

# Match the salary totals to C04's reported team payroll.
# validate="one_to_one" checks that franchise-season is unique
# in both datasets, preventing unexpected row multiplication.
payroll_audit = team_seasons[
    ["franchise_id", "season", "team_payroll"]
].merge(
    salary_totals,
    on=["franchise_id", "season"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Check whether every franchise-season has salary records.
print("Franchise-season matching:")
print(payroll_audit["_merge"].value_counts())

# Inspect the distribution of observed salary totals and payroll.
print("\nFinancial summary:")
display(
    payroll_audit[
        ["team_payroll", "observed_salary_total", "salary_record_count"]
    ].describe()
)


Franchise-season matching:
_merge
both          776
left_only       0
right_only      0
Name: count, dtype: int64

Financial summary:


,team_payroll,observed_salary_total,salary_record_count
count,7.760000e+02,7.760000e+02,776.000000
mean,9.408558e+07,8.818326e+07,15.239691
std,4.245115e+07,4.642792e+07,4.088918
min,2.392258e+07,6.000000e+06,1.000000
25%,6.238166e+07,5.614665e+07,14.000000
50%,7.599285e+07,7.290948e+07,16.000000
75%,1.241759e+08,1.177673e+08,18.000000
max,2.207089e+08,2.353917e+08,28.000000


In [15]:

# F2 | Step 3B — Calculate salary-to-payroll reconciliation ratios.

# Divide each franchise-season's summed C07 salaries by C04 payroll.
# Multiplying by 100 expresses the result as a percentage.
payroll_audit["salary_to_payroll_pct"] = (
    payroll_audit["observed_salary_total"]
    / payroll_audit["team_payroll"]
    * 100
)

# Summarize the distribution across all franchise-seasons.
display(
    payroll_audit["salary_to_payroll_pct"]
    .describe()
    .round(1)
)

# Count how many franchise-seasons fall into useful audit bands.
print("Below 80%:",
      (payroll_audit["salary_to_payroll_pct"] < 80).sum())

print("80% to 120%:",
      payroll_audit["salary_to_payroll_pct"].between(80, 120).sum())

print("Above 120%:",
      (payroll_audit["salary_to_payroll_pct"] > 120).sum())

# Inspect the franchise-seasons with the lowest reconciliation.
display(
    payroll_audit[
        ["franchise_id", "season", "salary_record_count",
         "team_payroll", "observed_salary_total",
         "salary_to_payroll_pct"]
    ]
    .sort_values("salary_to_payroll_pct")
    .head(10)
)


count    776.0
mean      91.7
std       19.0
min       11.0
25%       84.9
50%       96.6
75%      101.4
max      176.3
Name: salary_to_payroll_pct, dtype: float64

Below 80%: 145
80% to 120%: 609
Above 120%: 22


,franchise_id,season,salary_record_count,team_payroll,observed_salary_total,salary_to_payroll_pct
121,1610612765,2004-2005,1,54762481,6000000,10.956406
145,1610612740,2004-2005,3,56572882,7864000,13.900653
140,1610612744,2004-2005,3,54943723,7713000,14.038000
127,1610612764,2004-2005,2,49797050,8404000,16.876502
135,1610612762,2004-2005,2,43515760,8047000,18.492151
90,1610612738,2003-2004,4,60185915,13307000,22.109824
116,1610612738,2004-2005,1,64577356,14625000,22.647257
123,1610612754,2004-2005,3,66064780,15606000,23.622269
400,1610612753,2013-2014,8,79294567,19386125,24.448239
122,1610612741,2004-2005,2,57276129,15150000,26.450810


In [16]:

# F2 | Step 3C — Summarize payroll reconciliation by season.

season_payroll_audit = (
    payroll_audit
    .groupby("season")
    .agg(
        franchise_seasons=("franchise_id", "size"),

        # Median is less sensitive to extreme team-level discrepancies.
        median_reconciliation_pct=("salary_to_payroll_pct", "median"),

        # Count teams with particularly low observed salary totals.
        teams_below_80_pct=(
            "salary_to_payroll_pct",
            lambda values: (values < 80).sum()
        ),

        # Typical number of observed salary records per franchise.
        median_salary_records=("salary_record_count", "median")
    )
    .round(1)
)

display(season_payroll_audit)


,franchise_seasons,median_reconciliation_pct,teams_below_80_pct,median_salary_records
season,,,,
2000-2001,29,87.3,5,13.0
2001-2002,29,97.0,5,14.0
2002-2003,29,78.7,15,9.0
2003-2004,29,60.5,26,6.0
2004-2005,30,36.4,26,4.0
2005-2006,30,95.0,5,14.0
2006-2007,30,98.1,4,15.0
2007-2008,30,97.2,4,15.5
2008-2009,30,97.9,3,15.0



### F2 — Steps 1–3: Schema, Grain, and Salary Coverage

- C07: 11,826 salary records; C05: 12,810 player-team-season records.
- Both tables have unique `(player_id, franchise_id, season)` keys.
- Direct matching: 10,601 records (82.8% of C05; 89.6% of C07).
- Unmatched: 2,209 C05-only and 1,225 C07-only records.
- Most unmatched records do not have a corresponding player-season in the other table.
- Salary records exist for all 776 franchise-seasons.
- Median C07 salary-total / C04 payroll reconciliation: 96.6%.
- 145 franchise-seasons reconcile below 80%; 22 exceed 120%.
- 2002–03 through 2004–05 show severe salary undercoverage.
- Recent seasons (2023–24 through 2025–26) show median payroll reconciliation of approximately 97–99%.
- Record-count matching and dollar-value reconciliation measure different types of coverage.
- Salary concentration features require franchise-season quality checks.
- Salary and reported payroll are not assumed to be accounting-equivalent.



### F2 — Additional findings

Manual review of ESPN's historical salary pages indicates that
2003–04 and 2004–05 expose approximately 3–4 pages of records,
versus roughly 12–15 pages in better-covered seasons.

This is consistent with the severe salary undercoverage identified
in the C07-to-C05 matching and C07-to-C04 payroll reconciliation audits.

Working diagnosis: incomplete upstream ESPN salary coverage.
This has not been proven to be the sole cause of unmatched records.

Implications:
- Preserve these seasons for team-level financial analysis where C04 is reliable.
- Flag 2002–03 through 2004–05 for salary-distribution analyses.
- Do not interpret observed salary shares as complete roster shares.
- Apply franchise-season quality checks before computing concentration features.
- No paid data enrichment is planned at this stage.



In [17]:

# F2 | Step 4A — Identify available financial denominators.

print("C01 — Seasons")
print(seasons.columns.tolist())

print("\nC04 — Team Seasons")
print([
    col for col in team_seasons.columns
    if any(term in col.lower() for term in ["payroll", "salary", "tax", "apron"])
])


C01 — Seasons
['season', 'Year', 'Salary Cap', 'Salary Cap Floor Pct', 'Salary Cap Floor', 'Luxury Tax', '1st Apron', '2nd Apron', 'BAE', 'Non-Taxpayer MLE', 'Taxpayer MLE', 'Team Room MLE', 'cba_id']

C04 — Team Seasons
['team_payroll', 'salary_cap_hit', 'salary_cap_space', 's18_tax_threshold', 'taxable_payroll', 'tax_space', 'tax_paid', 'estimated_tax_bill', 'active_salary', 'dead_salary', 'has_luxury_tax_source']


In [18]:

# F2 | Step 4B — Test financial denominator joins.

# Select only the C01 columns needed for salary normalization.
season_denominators = seasons[
    [
        "season",
        "Salary Cap",
        "Salary Cap Floor",
        "Luxury Tax",
        "1st Apron",
        "2nd Apron",
        "cba_id"
    ]
]

# Select the C04 payroll denominator.
team_denominators = team_seasons[
    ["franchise_id", "season", "team_payroll"]
]

# Start with C07, preserving every player salary observation.
f2_normalization_audit = (
    player_salaries
    .merge(
        season_denominators,
        on="season",
        how="left",
        validate="many_to_one"
    )
    .merge(
        team_denominators,
        on=["franchise_id", "season"],
        how="left",
        validate="many_to_one"
    )
)

# Verify that the joins did not change the number of salary records.
print("Original C07 rows:", len(player_salaries))
print("Joined audit rows:", len(f2_normalization_audit))

# Check whether every salary record received a valid cap and payroll.
print("\nMissing salary cap:", f2_normalization_audit["Salary Cap"].isna().sum())
print("Missing team payroll:", f2_normalization_audit["team_payroll"].isna().sum())

# Preview a few records to verify that denominators were attached.
display(
    f2_normalization_audit[
        [
            "player_id", "franchise_id", "season",
            "salary", "Salary Cap", "team_payroll"
        ]
    ].head(5)
)


Original C07 rows: 11826
Joined audit rows: 11826

Missing salary cap: 0
Missing team payroll: 0


,player_id,franchise_id,season,salary,Salary Cap,team_payroll
0,389,1610612737,2000-2001,7167000,35500000,39297129
1,673,1610612737,2000-2001,5909000,35500000,39297129
2,953,1610612737,2000-2001,4950000,35500000,39297129
3,1074,1610612737,2000-2001,548000,35500000,39297129
4,1510,1610612737,2000-2001,4250000,35500000,39297129


In [19]:

# F2 | Step 4C — Audit the feasibility of salary normalization.

denominator_columns = [
    "Salary Cap",
    "Salary Cap Floor",
    "Luxury Tax",
    "1st Apron",
    "2nd Apron",
    "team_payroll"
]

normalization_results = []

for col in denominator_columns:

    # Count salary observations with a valid, positive denominator.
    valid_count = (
        f2_normalization_audit[col].notna()
        & (f2_normalization_audit[col] > 0)
    ).sum()

    # Count unavailable denominators separately from invalid values.
    missing_count = f2_normalization_audit[col].isna().sum()

    invalid_count = (
        f2_normalization_audit[col].notna()
        & (f2_normalization_audit[col] <= 0)
    ).sum()

    # Store one summary record per denominator.
    normalization_results.append({
        "denominator": col,
        "valid_records": valid_count,
        "missing_records": missing_count,
        "zero_or_negative_records": invalid_count,
        "coverage_pct": round(
            valid_count / len(f2_normalization_audit) * 100, 1
        )
    })

# Convert the list of dictionaries into a readable audit table.
normalization_summary = pd.DataFrame(normalization_results)

display(normalization_summary)


,denominator,valid_records,missing_records,zero_or_negative_records,coverage_pct
0,Salary Cap,11826,0,0,100.0
1,Salary Cap Floor,11826,0,0,100.0
2,Luxury Tax,10925,901,0,92.4
3,1st Apron,7679,4147,0,64.9
4,2nd Apron,1433,10393,0,12.1
5,team_payroll,11826,0,0,100.0



### F2 — Step 4: Salary Normalization Feasibility

- C07 successfully joins to C01 season thresholds and C04 franchise payroll.
- All 11,826 salary records are preserved without row multiplication.
- Salary cap, salary floor, and team payroll denominators have 100% valid coverage.
- Luxury-tax threshold coverage: 92.4%.
- First-apron threshold coverage: 64.9%.
- Second-apron threshold coverage: 12.1%.
- No zero or negative denominators were identified.
- Missing historical thresholds must remain null, not zero.
- CBA-specific threshold definitions and rule applicability require care.
- Salary normalization is feasible across eras, with era-specific restrictions.
- Player salary / team payroll is a descriptive payroll-share measure,
  not necessarily an official CBA salary-accounting measure.


In [20]:

# F2 | Step 5A — Test salary concentration on observed C07 records.
# This is an audit calculation, not a final feature table.

# Choose one well-reconciled franchise-season as a test case.
test_team = (
    payroll_audit.loc[
        payroll_audit["salary_to_payroll_pct"].between(95, 105)
        & (payroll_audit["salary_record_count"] >= 14)
    ]
    .iloc[0]
)

test_franchise = test_team["franchise_id"]
test_season = test_team["season"]

# Select observed salaries for that franchise-season.
test_salaries = player_salaries.loc[
    (player_salaries["franchise_id"] == test_franchise)
    & (player_salaries["season"] == test_season)
].copy()

# Rank salaries from highest to lowest.
test_salaries = test_salaries.sort_values(
    "salary", ascending=False
)

observed_total = test_salaries["salary"].sum()

# Calculate top-3 salary share of observed salary total.
top3_share = (
    test_salaries["salary"].head(3).sum()
    / observed_total
)

# HHI is the sum of squared individual salary shares.
# Higher values indicate more concentrated observed compensation.
salary_hhi = (
    (test_salaries["salary"] / observed_total) ** 2
).sum()

print("Franchise:", test_franchise)
print("Season:", test_season)
print("Salary records:", len(test_salaries))
print(f"Top-3 observed salary share: {top3_share:.1%}")
print(f"Observed salary HHI: {salary_hhi:.4f}")

display(
    test_salaries[
        ["player_id", "source_player_name", "salary"]
    ].head(10)
)


Franchise: 1610612748
Season: 2000-2001
Salary records: 14
Top-3 observed salary share: 53.7%
Observed salary HHI: 0.1360


,player_id,source_player_name,salary
135,297,Alonzo Mourning,16879000
136,896,Tim Hardaway,12000000
133,224,Eddie Jones,8960000
134,258,Brian Grant,8897000
132,193,Anthony Mason,5553000
130,105,Dan Majerle,4000000
131,190,Duane Causwell,4000000
129,76,Cedric Ceballos,3937000
137,920,A.C. Green,2250000
141,1853,Anthony Carter,1200000


In [21]:

# F2 | Step 5B — Demonstrate sensitivity to incomplete salary coverage.

sensitivity_results = []

# Compare the complete observed 14-player sample with smaller subsets.
for n in [14, 10, 6, 4]:

    # Keep the n highest observed salaries.
    subset = test_salaries.head(n)

    # Recalculate the denominator using only retained salaries.
    subset_total = subset["salary"].sum()

    top3_share = subset["salary"].head(3).sum() / subset_total

    hhi = ((subset["salary"] / subset_total) ** 2).sum()

    sensitivity_results.append({
        "observed_players": n,
        "top3_share_pct": round(top3_share * 100, 1),
        "salary_hhi": round(hhi, 4)
    })

display(pd.DataFrame(sensitivity_results))


,observed_players,top3_share_pct,salary_hhi
0,14,53.7,0.1360
1,10,55.9,0.1470
2,6,67.2,0.2005
3,4,81.0,0.2694


In [22]:

# F2 | Step 5C — Test salary and performance integration.

performance_columns = [
    "player_id",
    "franchise_id",
    "season",
    "games_played",
    "points_total",
    "points_per_game",
    "true_shooting_pct",
    "usage_pct",
    "player_impact_estimate"
]

# Retain all C07 salary records and attach available C05 performance.
salary_performance_audit = player_salaries.merge(
    player_team_seasons[performance_columns],
    on=["player_id", "franchise_id", "season"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Confirm the number of matched and unmatched salary observations.
print("Original C07 rows:", len(player_salaries))
print("Joined rows:", len(salary_performance_audit))

print("\nMatch status:")
print(salary_performance_audit["_merge"].value_counts())

# Examine performance-field coverage among matched records only.
matched_performance = salary_performance_audit.loc[
    salary_performance_audit["_merge"] == "both"
]

print("\nPerformance coverage among matched records (%):")
print(
    matched_performance[
        performance_columns[3:]
    ]
    .notna()
    .mean()
    .mul(100)
    .round(1)
)


Original C07 rows: 11826
Joined rows: 11826

Match status:
_merge
both          10601
left_only      1225
right_only        0
Name: count, dtype: int64

Performance coverage among matched records (%):
games_played              100.0
points_total              100.0
points_per_game           100.0
true_shooting_pct         100.0
usage_pct                 100.0
player_impact_estimate    100.0
dtype: float64



### F2 — Step 5: Salary Concentration and Player Value

Salary concentration:
- Salary ranks, top-N salary shares, and salary HHI are computable.
- Miami 2000–01 test: 14 salary records; top-3 share 53.7%; HHI 0.1360.
- Removing lower-paid observations increased top-3 share and HHI.
- Concentration comparisons require salary-record counts and
  payroll-reconciliation diagnostics.
- Observed salary concentration is not necessarily complete roster concentration.

Salary versus performance:
- C07-to-C05 join preserved all 11,826 salary observations.
- 10,601 records matched (89.6% of C07).
- 1,225 salary observations had no exact C05 match.
- Six tested performance fields have 100% completeness among matches.
- Compensation-performance analysis is feasible on matched observations.
- Unmatched observations and selection bias must be documented.

Supporting datasets:
- C08 awards require aggregation before joining to player-season data.
- C09 rookie and minimum salary rules require eligibility and
  historical CBA applicability checks.

Decision:
- Salary ranking: Ready for observed records.
- Salary concentration: Conditional.
- Salary-performance comparisons: Ready for matched records.
- Awards and salary-rule comparisons: Conditional.



## F2 — Player Compensation & Value: Final Feasibility Decision

**Overall status: CONDITIONAL — core features ready; coverage-sensitive
and research-specific features require safeguards or further validation.**

### Sources
- C07: player_salaries
- C05: player_team_seasons
- C01: seasons
- C04: team_seasons
- C08: player_awards (future join validation)
- C09a/b: rookie_salary_scale, minimum_salary_rules (future validation)

### Validated findings
- C07 contains 11,826 salary records with unique
  (player_id, franchise_id, season) keys.
- C07 joins to C01 and C04 without row loss or multiplication.
- Salary cap, salary floor, and team payroll denominators: 100% valid.
- Luxury-tax threshold: 92.4% valid.
- First apron: 64.9% valid.
- Second apron: 12.1% valid.
- C07-to-C05 performance matches: 10,601 (89.6% of C07).
- All six tested performance fields are populated among matched rows.
- Salary rankings, top-N shares, and HHI are computable.

### Data-quality limitations
- Historical salary coverage is especially weak in 2002–03 to 2004–05.
- Salary records do not necessarily represent complete team rosters.
- C07 observed salary totals may differ from C04 team payroll
  because of missing records and/or accounting definitions.
- Salary concentration is sensitive to missing observations.
- Salary-performance analysis is restricted to matched records.
- Historical CBA threshold availability and regulatory meaning differ.

### Phase B engineering decisions
1. Build salary and cap-normalized compensation features.
2. Preserve observed salary record counts and payroll reconciliation.
3. Build salary ranks, top-N shares, and HHI with quality flags.
4. Build salary-performance features for matched observations.
5. Apply historical threshold availability and CBA-era rules.
6. Defer award-based and salary-rule-based features until needed.
7. Defer composite player-value scoring until methodology is defined.

### Analytical grain
- Player compensation: player_id + franchise_id + season
- Salary concentration: franchise_id + season
- League financial thresholds: season

### Final decision
Proceed to feature engineering after all feature-family audits.
No canonical tables are modified during this audit.


In [23]:

# F3 | Step 1 — Audit roster grain and coverage.
# Reuse player_team_seasons (C05) and team_seasons (C04).

roster_key = ["player_id", "franchise_id", "season"]
team_key = ["franchise_id", "season"]

# 1. Verify that each player-franchise-season occurs only once.
duplicate_count = player_team_seasons.duplicated(
    subset=roster_key
).sum()

# 2. Count distinct players observed for each franchise-season.
roster_counts = (
    player_team_seasons
    .groupby(team_key, as_index=False)
    .agg(
        observed_players=("player_id", "nunique"),
        total_player_games=("games_played", "sum")
    )
)

# 3. Check whether all C04 franchise-seasons appear in C05.
roster_coverage = team_seasons[team_key].merge(
    roster_counts,
    on=team_key,
    how="left",
    validate="one_to_one",
    indicator=True
)

# 4. Count player-seasons associated with multiple franchises.
multi_team_players = (
    player_team_seasons
    .groupby(["player_id", "season"])["franchise_id"]
    .nunique()
)

print("Duplicate player-franchise-season keys:", duplicate_count)
print("\nFranchise-season coverage:")
print(roster_coverage["_merge"].value_counts())

print("\nObserved players per franchise-season:")
print(roster_counts["observed_players"].describe().round(1))

print("\nPlayer-seasons involving multiple franchises:",
      (multi_team_players > 1).sum())

print("\nFranchise-seasons with the fewest observed players:")
display(roster_counts.nsmallest(5, "observed_players"))


Duplicate player-franchise-season keys: 0

Franchise-season coverage:
_merge
both          776
left_only       0
right_only      0
Name: count, dtype: int64

Observed players per franchise-season:
count    776.0
mean      16.5
std        2.2
min       12.0
25%       15.0
50%       16.0
75%       18.0
max       26.0
Name: observed_players, dtype: float64

Player-seasons involving multiple franchises: 0

Franchise-seasons with the fewest observed players:


,franchise_id,season,observed_players,total_player_games
7,1610612737,2007-2008,12,727
30,1610612738,2004-2005,12,831
54,1610612739,2002-2003,12,775
423,1610612753,2009-2010,12,839
424,1610612753,2010-2011,12,752


In [24]:

# F3 | Step 2 — Audit core roster feature feasibility.

# A. Age and participation coverage.
print("Age coverage (%):",
      round(player_team_seasons["age"].notna().mean() * 100, 1))

print("Games played coverage (%):",
      round(player_team_seasons["games_played"].notna().mean() * 100, 1))

print("Age range:",
      player_team_seasons["age"].min(),
      "to",
      player_team_seasons["age"].max())

print("Games played range:",
      player_team_seasons["games_played"].min(),
      "to",
      player_team_seasons["games_played"].max())

# B. Check player-season uniqueness independently of franchise.
# This tells us whether each player-season is represented once.
print("\nDuplicate player-season keys:",
      player_team_seasons.duplicated(
          subset=["player_id", "season"]
      ).sum())

# C. Test year-over-year continuity for one franchise.
# Sorting season labels works because they use YYYY-YYYY format.
test_franchise = player_team_seasons["franchise_id"].iloc[0]

franchise_rosters = (
    player_team_seasons.loc[
        player_team_seasons["franchise_id"] == test_franchise
    ]
    .groupby("season")["player_id"]
    .agg(set)
    .sort_index()
)

continuity_results = []

previous_roster = None

for season, current_roster in franchise_rosters.items():

    if previous_roster is not None:

        # Intersection identifies players assigned in both seasons.
        returning = len(current_roster & previous_roster)

        continuity_results.append({
            "season": season,
            "current_players": len(current_roster),
            "returning_players": returning,
            "retention_pct": round(
                returning / len(previous_roster) * 100, 1
            )
        })

    previous_roster = current_roster

display(pd.DataFrame(continuity_results).head(10))


Age coverage (%): 100.0
Games played coverage (%): 100.0
Age range: 18.0 to 44.0
Games played range: 1 to 85

Duplicate player-season keys: 0


,season,current_players,returning_players,retention_pct
0,2001-2002,17,5,31.2
1,2002-2003,14,9,52.9
2,2003-2004,17,5,35.7
3,2004-2005,15,5,29.4
4,2005-2006,17,7,46.7
5,2006-2007,16,13,76.5
6,2007-2008,19,7,43.8
7,2008-2009,15,7,36.8
8,2009-2010,15,11,73.3
9,2010-2011,16,6,40.0



## F3 — Roster Construction: Final Feasibility Decision

**Overall status: READY WITH QUALIFICATIONS**

### Primary source
- C05: player_team_seasons
- C04: team_seasons (franchise-season reference)
- C03: players (optional supporting attributes)

### Structural validation
- 12,810 player-franchise-season observations.
- Zero duplicate (player_id, franchise_id, season) keys.
- All 776 C04 franchise-seasons represented in C05.
- Observed players per franchise-season:
  - Mean: 16.5
  - Median: 16
  - Minimum: 12
  - Maximum: 26
- Zero duplicate (player_id, season) keys.
- Zero player-seasons assigned to multiple franchises.

### Field coverage
- Age: 100% populated; observed range 18–44.
- Games played: 100% populated; observed range 1–85.

### Feature feasibility

READY:
- Observed player count per franchise-season.
- Mean and median player age.
- Age distribution and age-band shares.
- Descriptive games-played summaries.

CONDITIONAL:
- Year-over-year roster retention.
- New-player share and roster turnover.
- Veteran/experience measures.
- Minutes-weighted roster composition.
- Player contribution aggregated to franchise-season.

DEFERRED:
- Actual trade or transaction counts.
- Opening-day and end-of-season roster composition.
- Exact midseason franchise-specific statistical attribution.

### Key limitations
- C05 represents at most one franchise per player-season.
- The canonical franchise-assignment convention is not yet verified.
- Player assignment changes do not identify transaction type or timing.
- Player-season statistics may not exclusively reflect activity
  with the assigned franchise.
- Roster continuity reflects canonical franchise assignments,
  not a complete transaction history.

### Engineering safeguards
- Preserve observed player counts alongside roster features.
- Define retention using consecutive franchise-season assignments.
- Distinguish previous-roster retention from current-roster
  returning-player share.
- Do not interpret assignment changes as verified trades.
- Verify statistical attribution before using player statistics
  as exact franchise-level contributions.
- Treat the first observed season as lacking prior-season
  continuity history.

### Final decision
Proceed with core roster-construction features in A01.
No additional source reconciliation required at this stage.


In [25]:

# F4 | Step 1 — Inspect leadership tenure structure.
# Reuse canonical_dir from our previous audits.

leadership_tenures = pd.read_csv(
    canonical_dir / "leadership_tenures.csv"
)

# A. Basic table dimensions and schema.
print("C06 shape:", leadership_tenures.shape)

print("\nC06 columns:")
print(leadership_tenures.columns.tolist())

# B. Check field completeness.
print("\nColumn completeness (%):")
print(
    leadership_tenures
    .notna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values()
)

# C. Check for completely identical records.
print("\nFully duplicated rows:",
      leadership_tenures.duplicated().sum())

# D. Preview the structure and actual values.
print("\nSample leadership records:")
display(leadership_tenures.head(10))


C06 shape: (251, 8)

C06 columns:
['franchise_id', 'leader_name', 'role', 'start_date', 'end_date', 'start_season', 'end_season', 'source_table']

Column completeness (%):
start_season     27.9
end_season       27.9
end_date         60.2
start_date       72.1
franchise_id    100.0
leader_name     100.0
role            100.0
source_table    100.0
dtype: float64

Fully duplicated rows: 0

Sample leadership records:


,franchise_id,leader_name,role,start_date,end_date,start_season,end_season,source_table
0,1610612737,Pete Babcock,executive,1990-02-14,2003-04-02,NaN,NaN,S15
1,1610612737,Billy Knight,executive,2003-04-02,2008-05-28,NaN,NaN,S15
2,1610612737,Rick Sund,executive,2008-05-28,2012-06-25,NaN,NaN,S15
3,1610612737,Danny Ferry,executive,2012-06-25,2014-09-12,NaN,NaN,S15
4,1610612737,Mike Budenholzer,executive,2014-09-12,2017-05-25,NaN,NaN,S15
5,1610612737,Travis Schlenk,executive,2017-05-25,2022-12-21,NaN,NaN,S15
6,1610612737,Landry Fields,executive,2022-12-21,2025-04-21,NaN,NaN,S15
7,1610612737,Onsi Saleh,executive,2025-04-21,NaN,NaN,NaN,S15
8,1610612738,Chris Wallace,executive,2001-01-08,2003-05-09,NaN,NaN,S15
9,1610612738,Danny Ainge,executive,2003-05-09,2021-06-02,NaN,NaN,S15


In [26]:

# F4 | Step 2 — Compare tenure formats by leadership role.

# Group records by role and summarize available tenure boundaries.
role_audit = (
    leadership_tenures
    .groupby("role")
    .agg(
        records=("leader_name", "size"),
        start_date_available=("start_date", "count"),
        end_date_available=("end_date", "count"),
        start_season_available=("start_season", "count"),
        end_season_available=("end_season", "count")
    )
)

display(role_audit)

# Check whether each record has at least one usable start boundary.
has_start = (
    leadership_tenures["start_date"].notna()
    | leadership_tenures["start_season"].notna()
)

# Check whether each record has at least one usable end boundary.
has_end = (
    leadership_tenures["end_date"].notna()
    | leadership_tenures["end_season"].notna()
)

print("\nRecords missing both start representations:",
      (~has_start).sum())

print("Records missing both end representations:",
      (~has_end).sum())

# Inspect which roles account for missing end boundaries.
print("\nMissing end boundaries by role:")
print(
    leadership_tenures.loc[~has_end, "role"]
    .value_counts()
)


,records,start_date_available,end_date_available,start_season_available,end_season_available
role,,,,,
executive,181,181,151,0,0
owner,70,0,0,70,70



Records missing both start representations: 0
Records missing both end representations: 30

Missing end boundaries by role:
role
executive    30
Name: count, dtype: int64



## F4 — Ownership & GM Behavior: Final Feasibility Decision

**Overall status: READY WITH QUALIFICATIONS**

### Primary sources
- C06: leadership_tenures
- C04: team_seasons
- C01: seasons
- F2/F3 analytical features when available

### Structural validation
- 251 leadership tenure records.
- 181 executive records and 70 ownership records.
- Zero fully duplicated rows.
- Franchise ID, leader name, and role: 100% complete.
- All records have a usable start boundary.

### Tenure representation
- Executives use calendar start/end dates.
- Owners use start/end seasons.
- Executive start dates: 181/181 populated.
- Executive end dates: 151/181 populated.
- Ownership start/end seasons: 70/70 populated.
- 30 executives have missing end dates.

### Interpretation
- Missing executive end dates are consistent with open-ended
  or ongoing appointments, but incumbency is not independently verified.
- Preserve missing end dates; do not invent departure dates.
- Owner end seasons may reflect source cutoffs rather than departures.

### Feature feasibility

READY:
- Leadership identities and role classifications.
- Source-record tenure starting points.

CONDITIONAL:
- Executive and ownership tenure duration.
- Leadership-to-franchise-season mapping.
- Leadership transitions and tenure flags.
- Spending and roster changes associated with leadership periods.
- Competitive outcomes during leadership periods.

DEFERRED:
- Causal attribution of individual executive decisions.
- Comprehensive historical front-office decision reconstruction.

### Engineering safeguards
- Convert date-based and season-based tenures using explicit rules.
- Define season-start or majority-of-season attribution.
- Flag seasons with leadership transitions.
- Preserve open-ended tenure indicators.
- Check for overlapping leadership assignments.
- Avoid multiplying A01 franchise-season records.
- Do not interpret associations as causal executive effects.

### Final decision
Proceed with leadership-era features during A01 engineering.
Detailed tenure-to-season mapping remains an engineering task.
No additional historical source reconciliation is required now.


In [27]:

# F5 | Step 1 — Audit CBA regimes and threshold applicability.

# Select the CBA identifier and financial thresholds from C01.
cba_audit = seasons[
    [
        "season",
        "cba_id",
        "Salary Cap",
        "Luxury Tax",
        "1st Apron",
        "2nd Apron"
    ]
].copy()

# A. Confirm the grain is one record per season.
print("Duplicate seasons:",
      cba_audit.duplicated(subset=["season"]).sum())

print("\nMissing CBA identifiers:",
      cba_audit["cba_id"].isna().sum())

# B. Count seasons assigned to each CBA regime.
print("\nSeasons per CBA regime:")
display(
    cba_audit
    .groupby("cba_id", dropna=False)
    .agg(
        first_season=("season", "min"),
        last_season=("season", "max"),
        season_count=("season", "size")
    )
    .reset_index()
)

# C. Display threshold availability for every season.
# This helps distinguish unavailable values from rules
# that did not exist or apply during earlier CBA eras.
print("\nSeason-by-season threshold availability:")
display(
    cba_audit.assign(
        cap_available=cba_audit["Salary Cap"].notna(),
        tax_available=cba_audit["Luxury Tax"].notna(),
        first_apron_available=cba_audit["1st Apron"].notna(),
        second_apron_available=cba_audit["2nd Apron"].notna()
    )[
        [
            "season",
            "cba_id",
            "cap_available",
            "tax_available",
            "first_apron_available",
            "second_apron_available"
        ]
    ]
)


Duplicate seasons: 0

Missing CBA identifiers: 0

Seasons per CBA regime:


,cba_id,first_season,last_season,season_count
0,cba_1999,2000-2001,2004-2005,5
1,cba_2005,2005-2006,2010-2011,6
2,cba_2011,2011-2012,2016-2017,6
3,cba_2017,2017-2018,2022-2023,6
4,cba_2023,2023-2024,2025-2026,3



Season-by-season threshold availability:


,season,cba_id,cap_available,tax_available,first_apron_available,second_apron_available
0,2000-2001,cba_1999,True,False,False,False
1,2001-2002,cba_1999,True,False,False,False
2,2002-2003,cba_1999,True,True,False,False
3,2003-2004,cba_1999,True,True,False,False
4,2004-2005,cba_1999,True,False,False,False
5,2005-2006,cba_2005,True,True,False,False
6,2006-2007,cba_2005,True,True,False,False
7,2007-2008,cba_2005,True,True,False,False
8,2008-2009,cba_2005,True,True,False,False
9,2009-2010,cba_2005,True,True,False,False


In [28]:

# F5 | Step 2 — Test second-apron classification.

# Select a season in which the second apron is defined.
test_season = "2024-2025"

# Join franchise financial records to season-level thresholds.
apron_test = (
    team_seasons.loc[
        team_seasons["season"] == test_season,
        ["franchise_id", "season", "team_payroll"]
    ]
    .merge(
        seasons[["season", "2nd Apron"]],
        on="season",
        how="left",
        validate="many_to_one"
    )
)

# Compute the difference between recorded payroll and the apron.
# Positive values indicate payroll above the threshold.
apron_test["second_apron_distance"] = (
    apron_test["team_payroll"] - apron_test["2nd Apron"]
)

# Preserve missing values instead of classifying unknowns as False.
apron_test["above_second_apron"] = (
    apron_test["second_apron_distance"]
    .gt(0)
    .where(apron_test["second_apron_distance"].notna())
)

print("Franchise-seasons:", len(apron_test))

print("Missing payroll values:",
      apron_test["team_payroll"].isna().sum())

print("Missing second-apron thresholds:",
      apron_test["2nd Apron"].isna().sum())

print("\nAbove-second-apron classification:")
print(apron_test["above_second_apron"].value_counts(dropna=False))

display(
    apron_test[
        [
            "franchise_id",
            "team_payroll",
            "2nd Apron",
            "second_apron_distance",
            "above_second_apron"
        ]
    ]
    .sort_values("second_apron_distance", ascending=False)
    .head(10)
)


Franchise-seasons: 30
Missing payroll values: 0
Missing second-apron thresholds: 0

Above-second-apron classification:
above_second_apron
False    25
True      5
Name: count, dtype: int64


,franchise_id,team_payroll,2nd Apron,second_apron_distance,above_second_apron
25,1610612756,220708856,188931000.0,31777856.0,True
20,1610612750,203708244,188931000.0,14777244.0,True
1,1610612738,195348491,188931000.0,6417491.0,True
17,1610612747,192057940,188931000.0,3126940.0,True
2,1610612752,191906878,188931000.0,2975878.0,True
18,1610612743,185864258,188931000.0,-3066742.0,False
4,1610612749,182559432,188931000.0,-6371568.0,False
9,1610612748,176102077,188931000.0,-12828923.0,False
24,1610612742,174411922,188931000.0,-14519078.0,False
19,1610612746,174124752,188931000.0,-14806248.0,False


In [29]:

# F5: Compare existing C04 payroll measures for six 2024–25 teams.
# This is a feasibility check, NOT an official apron classification.

test_ids = {
    1610612756: "Phoenix",
    1610612750: "Minnesota",
    1610612738: "Boston",
    1610612747: "Lakers",
    1610612752: "Knicks",
    1610612749: "Milwaukee",
}

payroll_check = team_seasons.loc[
    (team_seasons["season"] == "2024-2025")
    & (team_seasons["franchise_id"].isin(test_ids)),
    [
        "franchise_id",
        "season",
        "team_payroll",
        "salary_cap_hit",
        "taxable_payroll",
    ],
].copy()

# Add readable team names for easier comparison.
payroll_check["team"] = payroll_check["franchise_id"].map(test_ids)

# Official 2024–25 second-apron threshold from C01.
second_apron = 188_931_000

# Calculate how far each existing payroll measure is from the threshold.
# These distances are diagnostic proxies, not validated apron distances.
for column in ["team_payroll", "salary_cap_hit", "taxable_payroll"]:
    payroll_check[f"{column}_distance"] = (
        payroll_check[column] - second_apron
    )

display(
    payroll_check[
        [
            "team",
            "team_payroll",
            "salary_cap_hit",
            "taxable_payroll",
            "team_payroll_distance",
            "taxable_payroll_distance",
        ]
    ].sort_values("team")
)


,team,team_payroll,salary_cap_hit,taxable_payroll,team_payroll_distance,taxable_payroll_distance
717,Boston,195348491,195348491,193098445.0,6417491,4167445.0
718,Knicks,191906878,191906878,188877651.0,2975878,-53349.0
733,Lakers,192057940,192057940,188260871.0,3126940,-670129.0
720,Milwaukee,182559432,182559432,181791983.0,-6371568,-7139017.0
736,Minnesota,203708244,203708244,202359195.0,14777244,13428195.0
741,Phoenix,220708856,220708856,214371340.0,31777856,25440340.0



### F5 — CBA Regime and Threshold Feasibility

**Status: Conditionally ready**

- CBA regime (`cba_id`) is complete for all 26 seasons.
- Salary cap thresholds cover 26/26 seasons; luxury tax 23/26;
  first apron 15/26; second apron 3/26.
- C01 thresholds can support season-level CBA regime and
  threshold features, subject to historical applicability.
- C04 `team_payroll` and `salary_cap_hit` are identical
  across all 776 franchise-seasons.
- Six-team 2024–25 diagnostic: `taxable_payroll` is populated
  for all six and is lower than `team_payroll` in each case.
- Comparing `team_payroll` to the second apron identifies
  five of six teams above the threshold; substituting
  `taxable_payroll` identifies three of six.
- Neither C04 payroll field has been validated as the
  official Apron Team Salary numerator. These comparisons
  are diagnostic proxies, not confirmed apron classifications.
- The upstream source, accounting definition, and snapshot
  timing of `taxable_payroll` require reconciliation.

**A01 implications:** CBA regime indicators and threshold
values are feasible. Official tax/apron exposure flags and
threshold-distance features remain conditional on validating
the applicable salary-accounting numerators.

**Deferred action:** Reconcile C04 payroll source lineage and
obtain/validate tax and apron salary measures after completing
the F1–F6 audit, documentation review, and Git commit.


In [30]:

# F6 — Step 1: Identify competitive outcome columns in C04.

# Keywords help us locate potential regular-season and postseason fields.
outcome_keywords = [
    "win", "loss", "pct", "seed", "rank",
    "playoff", "postseason", "round",
    "final", "champion", "record", "standing"
]

# Search column names without assuming the exact naming convention.
outcome_columns = [
    col for col in team_seasons.columns
    if any(keyword in col.lower() for keyword in outcome_keywords)
]

print("Potential competitive outcome columns:")
print(outcome_columns)

# Inspect data types and non-null coverage for the identified columns.
if outcome_columns:
    outcome_audit = team_seasons[outcome_columns].agg(
        ["count", "nunique"]
    ).T

    outcome_audit["coverage_pct"] = (
        outcome_audit["count"] / len(team_seasons) * 100
    ).round(1)

    display(outcome_audit)

    # Preview a few franchise-seasons to inspect actual values.
    display(
        team_seasons[
            ["franchise_id", "season"] + outcome_columns
        ].head(10)
    )


Potential competitive outcome columns:
['wins', 'losses', 'win_pct', 'league_rank', 'pythagorean_wins', 'pythagorean_losses', 'true_shooting_pct', 'seed', 'playoff_series_won', 'round_score', 'furthest_round', 'made_playoffs']


,count,nunique,coverage_pct
wins,776,60,100.0
losses,776,59,100.0
win_pct,776,118,100.0
league_rank,776,30,100.0
pythagorean_wins,776,55,100.0
pythagorean_losses,776,54,100.0
true_shooting_pct,776,120,100.0
seed,416,8,53.6
playoff_series_won,776,5,100.0
round_score,416,4,53.6


,franchise_id,season,wins,losses,win_pct,league_rank,pythagorean_wins,pythagorean_losses,true_shooting_pct,seed,playoff_series_won,round_score,furthest_round,made_playoffs
0,1610612755,2000-2001,56,26,0.683,4,54,28,0.518,1.0,3,4.0,NBA Finals,True
1,1610612748,2000-2001,50,32,0.610,10,48,34,0.515,3.0,0,1.0,First Round,True
2,1610612752,2000-2001,48,34,0.585,9,50,32,0.523,4.0,0,1.0,First Round,True
3,1610612753,2000-2001,43,39,0.524,16,44,38,0.513,7.0,0,1.0,First Round,True
4,1610612738,2000-2001,36,46,0.439,20,35,47,0.521,NaN,0,NaN,NaN,False
5,1610612751,2000-2001,26,56,0.317,24,26,56,0.504,NaN,0,NaN,NaN,False
6,1610612764,2000-2001,19,63,0.232,27,23,59,0.514,NaN,0,NaN,NaN,False
7,1610612749,2000-2001,52,30,0.634,7,52,30,0.542,2.0,2,3.0,Conference Finals,True
8,1610612761,2000-2001,47,35,0.573,11,48,34,0.510,5.0,1,2.0,Conference Semifinals,True
9,1610612766,2000-2001,46,36,0.561,13,48,34,0.506,6.0,1,2.0,Conference Semifinals,True


In [31]:

# F6 — Step 2: Validate internal consistency of competitive outcomes.

# Calculate expected winning percentage from the underlying record.
calculated_win_pct = (
    team_seasons["wins"]
    / (team_seasons["wins"] + team_seasons["losses"])
)

# Allow a small difference because source win_pct is rounded.
win_pct_mismatch = (
    (team_seasons["win_pct"] - calculated_win_pct).abs() > 0.001
)

# Non-playoff teams should not have playoff-specific information.
non_playoff = ~team_seasons["made_playoffs"]

# Playoff teams should have a seed and a recorded furthest round.
playoff = team_seasons["made_playoffs"]

checks = {
    "Winning percentage mismatches": win_pct_mismatch.sum(),
    "Non-playoff teams with a seed": (
        non_playoff & team_seasons["seed"].notna()
    ).sum(),
    "Non-playoff teams with a furthest round": (
        non_playoff & team_seasons["furthest_round"].notna()
    ).sum(),
    "Playoff teams missing a seed": (
        playoff & team_seasons["seed"].isna()
    ).sum(),
    "Playoff teams missing a furthest round": (
        playoff & team_seasons["furthest_round"].isna()
    ).sum(),
}

for check, count in checks.items():
    print(f"{check}: {count}")


Winning percentage mismatches: 0
Non-playoff teams with a seed: 0
Non-playoff teams with a furthest round: 0
Playoff teams missing a seed: 0
Playoff teams missing a furthest round: 0


In [32]:

# F6 — Step 3: Check competitive outcomes at the season level.

season_check = (
    team_seasons.groupby("season")
    .agg(
        teams=("franchise_id", "nunique"),
        total_wins=("wins", "sum"),
        total_losses=("losses", "sum"),
        playoff_teams=("made_playoffs", "sum"),
        max_wins=("wins", "max"),
        min_games=("wins", lambda x: 0),  # Replaced below with actual games
    )
    .reset_index()
)

# Calculate each team's games played, then inspect season-level ranges.
team_seasons["games_check"] = (
    team_seasons["wins"] + team_seasons["losses"]
)

games_by_season = (
    team_seasons.groupby("season")["games_check"]
    .agg(["min", "max"])
    .reset_index()
)

season_check = season_check.drop(columns="min_games").merge(
    games_by_season, on="season"
)

# The league should have one win and one loss for every game.
season_check["win_loss_difference"] = (
    season_check["total_wins"] - season_check["total_losses"]
)

display(season_check)


,season,teams,total_wins,total_losses,playoff_teams,max_wins,min,max,win_loss_difference
0,2000-2001,29,1189,1189,16,58,82,82,0
1,2001-2002,29,1189,1189,16,61,82,82,0
2,2002-2003,29,1189,1189,16,60,82,82,0
3,2003-2004,29,1189,1189,16,61,82,82,0
4,2004-2005,30,1230,1230,16,62,82,82,0
5,2005-2006,30,1230,1230,16,64,82,82,0
6,2006-2007,30,1230,1230,16,67,82,82,0
7,2007-2008,30,1230,1230,16,66,82,82,0
8,2008-2009,30,1230,1230,16,66,82,82,0
9,2009-2010,30,1230,1230,16,61,82,82,0


In [33]:

# F6 — Final check: Exactly one NBA champion per season.

championship_check = (
    team_seasons.assign(
        is_champion=team_seasons["playoff_series_won"].eq(4)
    )
    .groupby("season")["is_champion"]
    .sum()
)

print("Seasons with exactly one champion:",
      championship_check.eq(1).sum(), "/", len(championship_check))

print("Seasons with unexpected champion counts:")
display(championship_check[championship_check.ne(1)])


Seasons with exactly one champion: 26 / 26
Seasons with unexpected champion counts:


Series([], Name: is_champion, dtype: int64)

### F6 — Competitive Outcomes Feasibility Audit

**Status: PASS — Ready for analytical feature engineering**

**Scope:** C04 `team_seasons.csv`, 776 franchise-season records spanning 2000–01 through 2025–26.

**Coverage**
- All 776 franchise-seasons contain regular-season wins, losses, winning percentage, league rank, Pythagorean wins/losses, true shooting percentage, playoff qualification, and playoff series won.
- Playoff-specific fields (`seed`, `round_score`, `furthest_round`) are populated for 416 playoff teams (53.6% of records). Missing values for non-playoff teams are intentional.

**Validation results**
- Winning percentage inconsistencies: 0.
- Non-playoff teams with an assigned playoff seed: 0.
- Non-playoff teams with a recorded furthest round: 0.
- Playoff teams missing a seed or furthest round: 0.
- League-wide wins equal losses in all 26 seasons.
- Exactly 16 playoff teams in each season.
- Exactly one championship winner (four playoff series won) in each of the 26 seasons.

**Historical considerations**
- League membership increased from 29 to 30 franchises in 2004–05.
- The 2011–12 lockout season had 66 games per team.
- The 2019–20 COVID-disrupted season had variable team game counts.
- The 2020–21 season had 72 games per team.
- The 2012–13 season includes a team with 81 games.

**A01 implications**
- `win_pct` is preferable to raw wins for cross-season comparisons.
- `made_playoffs`, `playoff_series_won`, and `furthest_round` support postseason-outcome analysis.
- Wins minus Pythagorean wins can be derived as an exploratory performance feature.
- Competitive outcomes are suitable for descriptive analysis and modeling, with appropriate controls for season, league conditions, and differences in games played.

**Conclusion:** F6 passes feasibility validation. No additional source extraction or canonical-table modifications are required for this feature family.